<a href="https://colab.research.google.com/github/kurkervin16-cmd/siklab-net-software/blob/main/Signal2Noise_Software.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
import json, os, requests
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential, Input
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# ==========================================
# 1. DOWNLOAD CLEAN DATA
# ==========================================
print("Step 1: Downloading clean NASA POWER solar data...")
url = "https://power.larc.nasa.gov/api/temporal/hourly/point"
params = {
    "parameters": "ALLSKY_SFC_SW_DWN",
    "community": "RE",
    "longitude": 122.59, # Region 6 (Guimaras/Iloilo area)
    "latitude": 10.60,
    "start": "20150101",
    "end": "20241231",
    "format": "JSON"
}

r = requests.get(url, params=params, timeout=120)
raw_data = r.json()["properties"]["parameter"]["ALLSKY_SFC_SW_DWN"]

# Ensure directories exist
os.makedirs("data", exist_ok=True)
os.makedirs("model", exist_ok=True)

with open("data/solar_data.json", "w") as f:
    json.dump(raw_data, f)

# ==========================================
# 2. PREPROCESS DATA
# ==========================================
print("Step 2: Processing solar data...")
values = np.array([max(0, v) for v in raw_data.values()], dtype=np.float32)

X_MAX = values.max()
values_norm = values / X_MAX
np.save("model/x_max.npy", X_MAX)

WINDOW = 24
X, y = [], []
for i in range(WINDOW, len(values_norm) - 1):
    X.append(values_norm[i-WINDOW:i])
    y.append(values_norm[i])

X = np.array(X).reshape(-1, WINDOW, 1)
y = np.array(y)

split = int(len(X) * 0.9)
X_train, X_val = X[:split], X[split:]
y_train, y_val = y[:split], y[split:]

# ==========================================
# 3. BUILD WARNING-FREE MODEL (UNROLLED)
# ==========================================
print("Step 3: Training model...")
# Using Input(shape) kills the Keras 3 shape warning
# Setting unroll=True flattens the LSTM loop into static math, removing compiler issues
model = Sequential([
    Input(shape=(WINDOW, 1)),
    LSTM(64, return_sequences=True, unroll=True),
    Dropout(0.2),
    LSTM(32, unroll=True),
    Dropout(0.2),
    Dense(1)
])

model.compile(optimizer="adam", loss="mse", metrics=["mae"])

callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint("model/best_model.keras", save_best_only=True, verbose=1)
]

model.fit(X_train, y_train, epochs=15, batch_size=256, validation_data=(X_val, y_val), callbacks=callbacks, verbose=1)

# ==========================================
# 4. EXPORT TO PURE NATIVE TFLITE
# ==========================================
print("Step 4: Exporting native TFLite model...")
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# Because the LSTM is unrolled, we do NOT need SELECT_TF_OPS or experimental flags.
# This results in a tiny, fast, pure TFLite model.
tflite_model = converter.convert()

with open("model/siklab_model.tflite", "wb") as f:
    f.write(tflite_model)

print("\n=== SUCCESS: All files processed and saved cleanly inside 'model/' folder! ===")

Step 1: Downloading clean NASA POWER solar data...
Step 2: Processing solar data...
Step 3: Training model...
Epoch 1/15
308/309 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - loss: 0.0333 - mae: 0.1285
Epoch 1: val_loss improved from None to 0.00338, saving model to model/best_model.keras

Epoch 1: finished saving model to model/best_model.keras
309/309 ━━━━━━━━━━━━━━━━━━━━ 36s 84ms/step - loss: 0.0152 - mae: 0.0833 - val_loss: 0.0034 - val_mae: 0.0400
Epoch 2/15
308/309 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - loss: 0.0048 - mae: 0.0478
Epoch 2: val_loss improved from 0.00338 to 0.00235, saving model to model/best_model.keras

Epoch 2: finished saving model to model/best_model.keras
309/309 ━━━━━━━━━━━━━━━━━━━━ 26s 84ms/step - loss: 0.0044 - mae: 0.0454 - val_loss: 0.0024 - val_mae: 0.0321
Epoch 3/15
308/309 ━━━━━━━━━━━━━━━━━━━━ 0s 69ms/step - loss: 0.0034 - mae: 0.0384
Epoch 3: val_loss improved from 0.00235 to 0.00141, saving model to model/best_model.keras

Epoch 3: finished saving model to model

In [13]:
import numpy as np

# 1. Load your scaler from the model folder (run this at server startup)
try:
    x_max = np.load('model/x_max.npy')
except FileNotFoundError:
    x_max = 1.0  # Fallback if raw data wasn't normalized

def predict_next_step(data_window):
    """
    data_window: A list or array of the last 24 raw values
    """
    if len(data_window) != 24:
        raise ValueError(f"Model requires exactly 24 data points. Got {len(data_window)}")

    # 2. Normalize using your saved scaling factor
    normalized = np.array(data_window, dtype=np.float32) / x_max

    # 3. Reshape from (24,) to (1, 24, 1) to match the required TensorSpec
    model_input = normalized.reshape(1, 24, 1)

    # 4. Run inference
    prediction = model.predict(model_input, verbose=0)

    # Return the single scalar output
    return float(prediction[0][0])

In [14]:
# Next Cell: Verify the freshly compiled native TFLite model
import numpy as np
import tensorflow as tf

# 1. Load the scale coefficient and instantiate the TFLite Interpreter
X_MAX = float(np.load("model/x_max.npy"))
interpreter = tf.lite.Interpreter(model_path="model/siklab_model.tflite")
interpreter.allocate_tensors()

# Extract allocation details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Verified TFLite Input Spec Shape: {input_details[0]['shape']}")
print(f"Verified TFLite Output Spec Shape: {output_details[0]['shape']}\n")

# 2. Create a mock 24-hour raw solar window (simulating sunrise to sunset peaks)
mock_24hr_window = np.array([
    0.0, 0.0, 0.0, 0.0, 15.0, 85.0, 240.0, 450.0, 680.0, 850.0, 950.0, 980.0,
    920.0, 780.0, 550.0, 310.0, 120.0, 25.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0
], dtype=np.float32)

# 3. Normalize data and reshape to match the strict 3D TensorSpec (1, 24, 1)
normalized_input = (mock_24hr_window / X_MAX).reshape(1, 24, 1)

# 4. Bind input tensor data
interpreter.set_tensor(input_details[0]['index'], normalized_input)

# 5. Execute computation graph
interpreter.invoke()

# 6. Extract prediction array and invert normalization scaling
scaled_output = interpreter.get_tensor(output_details[0]['index'])[0][0]
predicted_wattage = scaled_output * X_MAX

print("📊 INFERENCE ENGINE TEST SUCCESSFUL:")
print(f"-> Normalized Signal Input: {normalized_input[0][-5:].flatten()}... (last 5 steps)")
print(f"-> Predicted Solar Generation for Hour 25: {predicted_wattage:.2f} W")

Verified TFLite Input Spec Shape: [ 1 24  1]
Verified TFLite Output Spec Shape: [1 1]

📊 INFERENCE ENGINE TEST SUCCESSFUL:
-> Normalized Signal Input: [0. 0. 0. 0. 0.]... (last 5 steps)
-> Predicted Solar Generation for Hour 25: -0.42 W


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [15]:
%%writefile decision_engine.py
import numpy as np
import tensorflow as tf
from simulator import set_relay

# Initialize the ultra-lightweight TFLite interpreter
interpreter = tf.lite.Interpreter(model_path="model/siklab_model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

X_MAX = float(np.load("model/x_max.npy"))

BATTERY_WH = 5000
SW_AMBER   = 4.0
SW_RED     = 2.0

def predict_next_hour(last_24):
    # Normalize and reshape to the exact (1, 24, 1) shape required
    data = (np.array(last_24, dtype=np.float32) / X_MAX).reshape(1, 24, 1)

    # Set tensor input and execute computational graph
    interpreter.set_tensor(input_details[0]['index'], data)
    interpreter.invoke()
    raw_pred = interpreter.get_tensor(output_details[0]['index'])[0][0]

    # Rescale back to true wattage and clamp at 0.0 to fix the negative nighttime bug
    return max(0.0, float(raw_pred) * X_MAX)

def compute_survival_window(soc_pct, iron_w):
    remaining_wh = BATTERY_WH * (soc_pct / 100)
    return round(remaining_wh / iron_w, 2) if iron_w > 0 else 99.0

def run_triage(sw):
    defer_keys    = ["defer_residential_a","defer_residential_b","defer_commercial"]
    priority_keys = ["priority_health","priority_gov"]
    if sw > SW_AMBER:
        for k in priority_keys + defer_keys: set_relay(k, True)
        return "NORMAL"
    elif sw > SW_RED:
        for k in priority_keys: set_relay(k, True)
        for k in defer_keys:    set_relay(k, False)
        return "CONSERVATION"
    else:
        for k in priority_keys + defer_keys: set_relay(k, False)
        return "CRITICAL"

Overwriting decision_engine.py


In [16]:
%%writefile app.py
from flask import Flask, render_template, request, jsonify
from flask_socketio import SocketIO
import threading, time
from simulator import snapshot, scenario
from decision_engine import predict_next_hour, compute_survival_window, run_triage

app      = Flask(__name__)
socketio = SocketIO(app, cors_allowed_origins="*")

step, irr_window = 0, [300.0] * 24
MODE_COLORS = {
    "NORMAL":       "#2ecc71",
    "CONSERVATION": "#f39c12",
    "CRITICAL":     "#e74c3c"
}

@app.route("/")
def index():
    return render_template("dashboard.html")

# WebSocket Event Handler (Bypasses localtunnel's proxy interception entirely)
@socketio.on("change_scenario")
def handle_change_scenario(data):
    scenario["type"] = data.get("event", "normal")
    print(f"⚠️ Scenario dynamically shifted to: {scenario['type']}")

def loop():
    global step, irr_window
    while True:
        snap        = snapshot(step)
        irr_window  = (irr_window + [snap["solar_w"]])[-24:]

        # Calls the updated TFLite decision engine
        predicted_w = predict_next_hour(irr_window)
        sw          = compute_survival_window(snap["battery_soc"], snap["iron_w"])
        mode        = run_triage(sw)

        socketio.emit("update", {
            **snap,
            "sw":          sw,
            "mode":        mode,
            "mode_color":  MODE_COLORS[mode],
            "predicted_w": round(predicted_w, 1),
        })
        step += 1
        time.sleep(3)

threading.Thread(target=loop, daemon=True).start()

if __name__ == "__main__":
    socketio.run(app, host="0.0.0.0", port=5000, debug=False, allow_unsafe_werkzeug=True)

Overwriting app.py


In [17]:
%%writefile simulator.py
import json, numpy as np, os
from datetime import datetime

# Safeguard: Load solar data if it exists, otherwise use a default clear-sky curve
if os.path.exists("data/solar_data.json"):
    with open("data/solar_data.json") as f:
        raw = json.load(f)
    irr_values = [max(0, v) for v in raw.values()]
else:
    # 24-hour baseline solar generation profile (sunrise to sunset)
    irr_values = [0.0, 0.0, 0.0, 0.0, 0.0, 10.0, 150.0, 400.0, 700.0, 900.0, 1000.0, 1050.0,
                  1000.0, 850.0, 600.0, 350.0, 100.0, 5.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]

relay_states = {
    "iron_vaccine":        True,
    "iron_water_pump":     True,
    "iron_comms":          True,
    "priority_health":     True,
    "priority_gov":        True,
    "defer_residential_a": True,
    "defer_residential_b": True,
    "defer_commercial":    True,
}

battery = {"soc_pct": 80.0, "voltage_v": 12.5, "capacity_wh": 5000}
scenario = {"type": "normal"}

def get_solar_w(step=0):
    if scenario["type"] == "drop":
        return 25.0
    return max(0, irr_values[step % len(irr_values)] * 0.5)

def get_loads():
    return {
        "iron_w":     220,
        "priority_w": 150 if relay_states["priority_health"] else 0,
        "defer_w":    300 if relay_states["defer_residential_a"] else 0,
    }

def update_battery(solar_w, total_w, dt_min=5):
    net_w    = solar_w - total_w
    delta_wh = net_w * (dt_min / 60)
    battery["soc_pct"] = max(0, min(100, battery["soc_pct"] + (delta_wh / battery["capacity_wh"]) * 100))
    battery["voltage_v"] = 11.5 + (battery["soc_pct"] / 100) * 1.2

def set_relay(name, state):
    if name.startswith("iron_"):
        return
    relay_states[name] = state

def snapshot(step=0):
    solar_w = get_solar_w(step)
    loads   = get_loads()
    update_battery(solar_w, sum(loads.values()))
    return {
        "solar_w":      round(solar_w, 1),
        "battery_soc":  round(battery["soc_pct"], 1),
        "battery_v":    round(battery["voltage_v"], 2),
        "iron_w":       loads["iron_w"],
        "priority_w":   loads["priority_w"],
        "defer_w":      loads["defer_w"],
        "relay_states": relay_states.copy(),
        "timestamp":    datetime.now().strftime("%H:%M:%S"),
    }

Overwriting simulator.py


In [18]:
# Cell: Create templates/dashboard.html
import os
os.makedirs("templates", exist_ok=True)

html = '''<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>SIKLAB Net</title>
  <script src="https://cdn.socket.io/4.5.4/socket.io.min.js"></script>
  <style>
    *{box-sizing:border-box;margin:0;padding:0}
    body{font-family:sans-serif;background:#0d1b2a;color:#fff;padding:24px}
    h1{font-size:16px;color:#b8d4f0;margin-bottom:20px}
    .sw-block{text-align:center;margin:20px 0}
    .sw-number{font-size:80px;font-weight:800;color:#f0a500;line-height:1}
    .sw-label{color:#888;font-size:12px;margin-top:6px}
    .mode-badge{display:inline-block;padding:8px 28px;border-radius:20px;
                font-weight:700;font-size:18px;margin:10px auto;transition:background .4s}
    .stats{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin:16px 0}
    .stat-card{background:#1a3a6b;border-radius:10px;padding:14px;text-align:center}
    .stat-card .label{font-size:10px;color:#888;margin-bottom:4px}
    .stat-card .value{font-size:22px;font-weight:700}
    .tiers{display:grid;grid-template-columns:repeat(3,1fr);gap:12px;margin:16px 0}
    .tier-card{border-radius:10px;padding:12px}
    .tier-card .tier-name{font-weight:700;font-size:12px;margin-bottom:8px}
    .relay-row{display:flex;justify-content:space-between;font-size:11px;margin:3px 0}
    .on{color:#2ecc71;font-weight:600}
    .off{color:#e74c3c;font-weight:600}
    .iron-card{background:rgba(192,57,43,.15);border:1px solid #c0392b}
    .priority-card{background:rgba(212,130,10,.15);border:1px solid #d4820a}
    .defer-card{background:rgba(46,125,50,.15);border:1px solid #2e7d32}
    .buttons{display:flex;gap:12px;justify-content:center;margin:20px 0}
    .btn{padding:12px 28px;font-size:14px;font-weight:600;border:none;
         border-radius:8px;cursor:pointer}
    .btn-storm{background:#e74c3c;color:#fff}
    .btn-restore{background:#2ecc71;color:#fff}
    .ts{text-align:center;color:#444;font-size:10px;margin-top:8px}
  </style>
</head>
<body>
  <h1>⚡ SIKLAB Net — Edge AI Grid Controller | Signal2Noise</h1>
  <div class="sw-block">
    <div class="sw-number" id="sw">--</div>
    <div class="sw-label">hours of critical infrastructure power remaining</div>
    <div><span class="mode-badge" id="mode" style="background:#2ecc71">NORMAL</span></div>
  </div>
  <div class="stats">
    <div class="stat-card">
      <div class="label">☀️ Solar Generation</div>
      <div class="value"><span id="solar">--</span> W</div>
    </div>
    <div class="stat-card">
      <div class="label">🔋 Battery SoC</div>
      <div class="value"><span id="batt">--</span> %</div>
    </div>
    <div class="stat-card">
      <div class="label">🔮 AI Forecast (next hr)</div>
      <div class="value"><span id="pred">--</span> W</div>
    </div>
  </div>
  <div class="tiers">
    <div class="tier-card iron-card">
      <div class="tier-name">🔴 IRON Tier</div>
      <div class="relay-row">Vaccine fridge <span id="r_vaccine">--</span></div>
      <div class="relay-row">Water pump <span id="r_pump">--</span></div>
      <div class="relay-row">Emergency comms <span id="r_comms">--</span></div>
    </div>
    <div class="tier-card priority-card">
      <div class="tier-name">🟡 PRIORITY Tier</div>
      <div class="relay-row">Health center <span id="r_health">--</span></div>
      <div class="relay-row">Gov office <span id="r_gov">--</span></div>
    </div>
    <div class="tier-card defer-card">
      <div class="tier-name">🟢 DEFERRABLE Tier</div>
      <div class="relay-row">Residential A <span id="r_resa">--</span></div>
      <div class="relay-row">Residential B <span id="r_resb">--</span></div>
      <div class="relay-row">Commercial <span id="r_com">--</span></div>
    </div>
  </div>
  <div class="buttons">
    <button class="btn btn-storm"   onclick="trigger(\'drop\')">☁️ Trigger Storm Event</button>
    <button class="btn btn-restore" onclick="trigger(\'normal\')">☀️ Restore Generation</button>
  </div>
  <div class="ts" id="ts">--</div>
<script>
  const socket = io();

  // Clean DOM swapping that doesn't break future selection queries
  function updateRelayDOM(elementId, state) {
    const el = document.getElementById(elementId);
    if (el) {
      el.textContent = state ? "ON" : "OFF";
      el.className = state ? "on" : "off";
    }
  }

  socket.on("update", d => {
    document.getElementById("sw").textContent    = d.sw.toFixed(1);
    document.getElementById("solar").textContent = d.solar_w;
    document.getElementById("batt").textContent  = d.battery_soc;
    document.getElementById("pred").textContent  = d.predicted_w;
    document.getElementById("ts").textContent    = "Last update: " + d.timestamp;

    const m = document.getElementById("mode");
    m.textContent = d.mode;
    m.style.background = d.mode_color;

    const rs = d.relay_states;
    updateRelayDOM("r_vaccine", rs.iron_vaccine);
    updateRelayDOM("r_pump", rs.iron_water_pump);
    updateRelayDOM("r_comms", rs.iron_comms);
    updateRelayDOM("r_health", rs.priority_health);
    updateRelayDOM("r_gov", rs.priority_gov);
    updateRelayDOM("r_resa", rs.defer_residential_a);
    updateRelayDOM("r_resb", rs.defer_residential_b);
    updateRelayDOM("r_com", rs.defer_commercial);
  });

  function trigger(e) {
    // Communicates via WebSocket to bypass Localtunnel HTTP proxy blocks
    socket.emit("change_scenario", {event: e});
  }
</script>
</body></html>'''

with open("templates/dashboard.html", "w") as f:
    f.write(html)
print("templates/dashboard.html created successfully with WebSocket trigger!")

templates/dashboard.html created successfully with WebSocket trigger!


In [19]:
# Next Cell: Automatically rewrite the frontend button trigger to use WebSockets
with open("templates/dashboard.html", "r") as f:
    html = f.read()

start_idx = html.find("function trigger")
if start_idx != -1:
    open_brace_idx = html.find("{", start_idx)
    brace_count = 1
    end_idx = open_brace_idx + 1

    while brace_count > 0 and end_idx < len(html):
        if html[end_idx] == "{":
            brace_count += 1
        elif html[end_idx] == "}":
            brace_count -= 1
        end_idx += 1

    old_function = html[start_idx:end_idx]
    new_function = """function trigger(e) {
    socket.emit("change_scenario", {event: e});
}"""

    html = html.replace(old_function, new_function)
    with open("templates/dashboard.html", "w") as f:
        f.write(html)
    print("SUCCESS: Frontend dashboard buttons re-routed to WebSockets! ✅")
else:
    print("ERROR: 'function trigger' not found in templates/dashboard.html.")

SUCCESS: Frontend dashboard buttons re-routed to WebSockets! ✅


In [20]:
# Next Cell: Complete runtime deployment
import subprocess, time, sys

# Drop stale compilation caches
for mod in ["decision_engine", "simulator", "app"]:
    if mod in sys.modules:
        del sys.modules[mod]

# Terminate any lingering background app instances
subprocess.run(["pkill", "-f", "app.py"], capture_output=True)
time.sleep(1)

# Boot the updated app.py
proc = subprocess.Popen(
    ["python", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# TFLite initializes immediately; 3 seconds is more than enough time to bind the port
time.sleep(3)

if proc.poll() is not None:
    _, err = proc.communicate()
    print("Process Crashed:\n", err.decode()[-500:])
else:
    import subprocess as sp
    check = sp.run(["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "http://localhost:5000"], capture_output=True, text=True)
    print(f"HTTP Local Status Code: {check.stdout}")

    if check.stdout == "200":
        print("SIKLAB Net is fully operational via TFLite engine! ⚡")
        from google.colab import output
        output.serve_kernel_port_as_iframe(5000, height=700)
    else:
        print("Port warming up. Wait 3 seconds and run this cell again.")

HTTP Local Status Code: 000
Port warming up. Wait 3 seconds and run this cell again.


In [21]:
# Cell: Panic Button - Kill everything and boot clean
import subprocess, time

print("🧹 Purging ghost Python processes and clearing port 5000...")
subprocess.run(["pkill", "-f", "app.py"], capture_output=True)
subprocess.run(["pkill", "-f", "localtunnel"], capture_output=True)
time.sleep(2)

print("🚀 Re-launching the TFLite backend server...")
proc = subprocess.Popen(
    ["python", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Give TensorFlow a solid 8 seconds to import and load into memory
print("⏳ Warming up port (Waiting 8 seconds for TensorFlow)...")
time.sleep(8)

# Check status
import subprocess as sp
check = sp.run(["curl", "-s", "-o", "/dev/null", "-w", "%{http_code}", "http://localhost:5000"], capture_output=True, text=True)
print(f"\n➔ HTTP Local Status Code: {check.stdout}")

if check.stdout == "200":
    print("✨ SIKLAB Net is back online and running cleanly! ⚡")
    from google.colab import output
    output.serve_kernel_port_as_iframe(5000, height=700)
else:
    print("❌ Port still locked or server crashed. Run this cell one more time.")

🧹 Purging ghost Python processes and clearing port 5000...
🚀 Re-launching the TFLite backend server...
⏳ Warming up port (Waiting 8 seconds for TensorFlow)...

➔ HTTP Local Status Code: 200
✨ SIKLAB Net is back online and running cleanly! ⚡


<IPython.core.display.Javascript object>

In [22]:
# Cell: Install WebSocket dependencies
!pip install flask-socketio simple-websocket